In [1]:
!nvidia-smi

Thu Mar 12 12:13:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
!python --version

Python 3.12.12


In [2]:
# !pip install vllm==0.8.5 transformers==4.57.3

# Embeddings

In [ ]:
import math
import torch
import torch.nn.functional as F
from vllm import SamplingParams
from vllm.inputs.data import TokensPrompt


def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery: {query}"


def attr_to_text(attr: dict) -> str:
    """
    Convert one KV attribute dict to text for embed/rerank.
    Keeps full dict in result, but text is based on `name` + `request_value`.
    """
    name = str(attr.get("name", "")).strip()
    value = str(attr.get("request_value", "")).strip()
    return f"{name}: {value}"


def format_instruction(instruction: str, query: str, doc: str):
    return [
        {
            "role": "system",
            "content": (
                'Judge whether the Document meets the requirements based on the Query '
                'and the Instruct provided. Note that the answer can only be "yes" or "no".'
            ),
        },
        {
            "role": "user",
            "content": f"<Instruct>: {instruction}\n\n<Query>: {query}\n\n<Document>: {doc}",
        },
    ]


def process_rerank_pair(
    tokenizer,
    query_text: str,
    doc_text: str,
    instruction: str,
    max_length: int,
    suffix_tokens: list[int],
):
    messages = [format_instruction(instruction, query_text, doc_text)]
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    tokenized = tokenized[0][:max_length] + suffix_tokens
    return TokensPrompt(prompt_token_ids=tokenized)


def compute_yes_probability(
    rerank_model,
    prompt,
    sampling_params,
    true_token: int,
    false_token: int,
) -> float:
    outputs = rerank_model.generate([prompt], sampling_params, use_tqdm=False)
    final_logits = outputs[0].outputs[0].logprobs[-1]

    true_logit = final_logits[true_token].logprob if true_token in final_logits else -10
    false_logit = final_logits[false_token].logprob if false_token in final_logits else -10

    true_score = math.exp(true_logit)
    false_score = math.exp(false_logit)
    return true_score / (true_score + false_score)


def evaluate_attrs_match(
    attrs_gt: list[dict],
    attrs_pred: list[dict],
    emb_model,
    rerank_model,
    tokenizer,
    emb_task: str,
    rer_task: str,
    match_threshold: float = 0.8,
    max_length: int = 8192,
):
    """
    Real evaluation for two attribute lists:
      - attrs_gt: ground truth attrs
      - attrs_pred: predicted attrs

    Logic:
      1) Build embeddings for GT as queries (with instruction) and PRED as documents.
      2) Compute pairwise similarity and sort all pairs by similarity desc.
      3) Iterate pairs in sorted order:
           - if gt or pred already matched -> skip
           - rerank this pair
           - if rerank score > threshold -> mark TP and consume both attrs
      4) Remaining predicted attrs -> FP
      5) Remaining ground-truth attrs -> FN

    Returns:
      list[dict] like:
        {"gt_attr": ..., "pred_attr": ..., "result": "TP", "sim": ..., "final_score": ...}
        {"gt_attr": None, "pred_attr": ..., "result": "FP", ...}
        {"gt_attr": ..., "pred_attr": None, "result": "FN", ...}
    """
    if not attrs_gt and not attrs_pred:
        return []

    # ---- prepare texts ----
    gt_texts_plain = [attr_to_text(a) for a in attrs_gt]
    pred_texts_plain = [attr_to_text(a) for a in attrs_pred]

    gt_texts_embed = [get_detailed_instruct(emb_task, t) for t in gt_texts_plain]
    pred_texts_embed = pred_texts_plain

    # ---- embeddings ----
    emb_inputs = gt_texts_embed + pred_texts_embed
    emb_outputs = emb_model.embed(emb_inputs)
    embeddings = torch.tensor([o.outputs.embedding for o in emb_outputs], dtype=torch.float32)
    embeddings = F.normalize(embeddings, p=2, dim=1)

    gt_emb = embeddings[: len(attrs_gt)]
    pred_emb = embeddings[len(attrs_gt) :]

    # pairwise cosine similarity
    sim_matrix = gt_emb @ pred_emb.T  # [n_gt, n_pred]

    # ---- all candidate pairs sorted by similarity desc ----
    candidate_pairs = []
    for i in range(len(attrs_gt)):
        for j in range(len(attrs_pred)):
            candidate_pairs.append((i, j, float(sim_matrix[i, j])))

    candidate_pairs.sort(key=lambda x: x[2], reverse=True)

    # ---- reranker config ----
    suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token

    true_token = tokenizer("yes", add_special_tokens=False).input_ids[0]
    false_token = tokenizer("no", add_special_tokens=False).input_ids[0]

    sampling_params = SamplingParams(
        temperature=0,
        max_tokens=1,
        logprobs=20,
        allowed_token_ids=[true_token, false_token],
    )

    # ---- greedy matching with rerank ----
    matched_gt = set()
    matched_pred = set()
    results = []

    for gt_idx, pred_idx, sim in candidate_pairs:
        if gt_idx in matched_gt or pred_idx in matched_pred:
            continue

        query_text = gt_texts_plain[gt_idx]
        doc_text = pred_texts_plain[pred_idx]

        prompt = process_rerank_pair(
            tokenizer=tokenizer,
            query_text=query_text,
            doc_text=doc_text,
            instruction=rer_task,
            max_length=max_length - len(suffix_tokens),
            suffix_tokens=suffix_tokens,
        )

        final_score = compute_yes_probability(
            rerank_model=rerank_model,
            prompt=prompt,
            sampling_params=sampling_params,
            true_token=true_token,
            false_token=false_token,
        )

        if final_score > match_threshold:
            matched_gt.add(gt_idx)
            matched_pred.add(pred_idx)
            results.append(
                {
                    "gt_attr": attrs_gt[gt_idx],
                    "pred_attr": attrs_pred[pred_idx],
                    "result": "TP",
                    "sim": sim,
                    "final_score": float(final_score),
                }
            )

    # ---- unmatched predicted -> FP ----
    for pred_idx, pred_attr in enumerate(attrs_pred):
        if pred_idx not in matched_pred:
            results.append(
                {
                    "gt_attr": None,
                    "pred_attr": pred_attr,
                    "result": "FP",
                    "sim": None,
                    "final_score": None,
                }
            )

    # ---- unmatched gt -> FN ----
    for gt_idx, gt_attr in enumerate(attrs_gt):
        if gt_idx not in matched_gt:
            results.append(
                {
                    "gt_attr": gt_attr,
                    "pred_attr": None,
                    "result": "FN",
                    "sim": None,
                    "final_score": None,
                }
            )

    return results



INFO 03-09 20:00:42 [__init__.py:239] Automatically detected platform cuda.


In [ ]:
from vllm import LLM
from transformers import AutoTokenizer

# embedding model
emb_model = LLM(
    model="Qwen/Qwen3-Embedding-4B",
    task="embed"
)

# reranker model
rerank_model = LLM(
    model="Qwen/Qwen3-Reranker-4B",
    tensor_parallel_size=torch.cuda.device_count(),
    max_model_len=5000,
    enable_prefix_caching=True,
    gpu_memory_utilization=0.7
)

# tokenizer ONLY for reranker
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Reranker-4B")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

INFO 03-09 20:01:12 [config.py:466] Found sentence-transformers modules configuration.


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 03-09 20:02:22 [loader.py:458] Loading weights took 2.31 seconds
INFO 03-09 20:02:22 [model_runner.py:1140] Model loading took 7.5441 GiB and 28.683766 seconds
INFO 03-09 20:02:24 [worker.py:287] Memory profiling takes 1.01 seconds
INFO 03-09 20:02:24 [worker.py:287] the current vLLM instance can use total_gpu_memory (39.49GiB) x gpu_memory_utilization (0.70) = 27.65GiB
INFO 03-09 20:02:24 [worker.py:287] model weights take 7.54GiB; non_torch_memory takes 0.02GiB; PyTorch activation peak memory takes 1.41GiB; the rest of the memory reserved for KV Cache is 18.68GiB.
INFO 03-09 20:02:24 [executor_base.py:112] # cuda blocks: 8500, # CPU blocks: 1820
INFO 03-09 20:02:24 [executor_base.py:117] Maximum concurrency for 5000 tokens per request: 27.20x
INFO 03-09 20:02:26 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 03-09 20:03:00 [model_runner.py:1592] Graph capturing finished in 33 secs, took 0.37 GiB
INFO 03-09 20:03:00 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 37.42 seconds


In [ ]:
!nvidia-smi

Mon Mar  9 20:03:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             59W /  400W |   35544MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm


def compute_prf(results):
    """
    Compute Precision, Recall, F1 from evaluation results.

    results: list of dicts like
      {"gt_attr": ..., "pred_attr": ..., "result": "TP|FP|FN"}
    """

    tp = sum(1 for r in results if r["result"] == "TP")
    fp = sum(1 for r in results if r["result"] == "FP")
    fn = sum(1 for r in results if r["result"] == "FN")

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def load_attrs(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return data


def evaluate_folder(
    json_dir,
    emb_model,
    rerank_model,
    tokenizer,
    emb_task,
    rer_task,
    out_xlsx="agg_results.xlsx",
    match_threshold=0.9,
    debug_dir='debug_results'
):
    json_dir = Path(json_dir)
    debug_dir = Path(debug_dir)
    debug_dir.mkdir(exist_ok=True)

    gt_files = {}
    pred_files = []

    for p in json_dir.glob("*.json"):
        stem = p.stem
        doc_id, strategy = stem.split("_", 1)

        if strategy == "gt":
            gt_files[doc_id] = p
        else:
            pred_files.append((doc_id, strategy, p))

    rows = []

    for doc_id, strategy, pred_path in tqdm(sorted(pred_files)):
        if doc_id not in gt_files:
            print(f"GT missing for {pred_path}")
            continue

        gt_path = gt_files[doc_id]

        attrs_gt = load_attrs(gt_path)
        attrs_pred = load_attrs(pred_path)

        results = evaluate_attrs_match(
            attrs_gt=attrs_gt,
            attrs_pred=attrs_pred,
            emb_model=emb_model,
            rerank_model=rerank_model,
            tokenizer=tokenizer,
            match_threshold=match_threshold,
            emb_task=emb_task,
            rer_task=rer_task
        )
        with open(debug_dir / pred_path.name, 'w') as f:
            f.write(json.dumps(results, ensure_ascii=False))

        metrics = compute_prf(results)
        print(pred_path.stem)
        print(metrics)

        rows.append(
            {
                "DOC_NAME": int(doc_id),
                "STRATEGY": strategy,
                "PAIR_F1": metrics["f1"],
                "PAIR_REC": metrics["recall"],
                "PAIR_PRC": metrics["precision"],
                "TP": metrics["TP"],
                "FP": metrics["FP"],
                "FN": metrics["FN"],
            }
        )

    df = (
        pd.DataFrame(rows)
        .sort_values(["DOC_NAME", "STRATEGY"])
        .reset_index(drop=True)
    )

    df[["DOC_NAME", "STRATEGY", "PAIR_F1", "PAIR_REC", "PAIR_PRC"]].to_csv(
        debug_dir / out_xlsx.replace('.xlsx', '.csv'),
        index=False
      )

    with pd.ExcelWriter(debug_dir / out_xlsx) as writer:
        df[["DOC_NAME", "STRATEGY", "PAIR_F1", "PAIR_REC", "PAIR_PRC"]]\
            .to_excel(writer, index=False, sheet_name="eval_table_docs_strategies")

        (
            df.groupby("STRATEGY")[["PAIR_F1", "PAIR_REC", "PAIR_PRC"]]
            .mean()
            .sort_values("PAIR_F1", ascending=False)
            .to_excel(writer, sheet_name="strategy_avg")
        )

    return df


In [ ]:
# !unzip -qq gt_old.zip

In [ ]:
rer_task = """Decide whether two requirement/characteristic name and values refer to the SAME requirement.
- "yes" if Query and Document are paraphrases / same meaning / same field, even with minor formatting or punctuation changes.
- "no" if meaning differs, scope differs, or one is only a partial subset of the other (missing essential clauses), diffirent in wording small but significant."""

emb_task = "Given a characteristic search query, retrieve relevant characteristic that pure matches name and value of the query characteristic"


df = evaluate_folder(
    json_dir="gt_old",
    emb_model=emb_model,
    rerank_model=rerank_model,
    tokenizer=tokenizer,
    rer_task=rer_task,
    emb_task=emb_task,
    out_xlsx="./agg_results.xlsx",
    match_threshold=0.9,
    debug_dir='gt_old_debug'
)

df


  0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/174 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  2%|▏         | 1/50 [00:07<06:01,  7.38s/it]

1_doc
{'TP': 79, 'FP': 6, 'FN': 10, 'precision': 0.9294117647058824, 'recall': 0.8876404494382022, 'f1': 0.9080459770114941}


Processed prompts:   0%|          | 0/174 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

9_row
{'TP': 62, 'FP': 28, 'FN': 12, 'precision': 0.6888888888888889, 'recall': 0.8378378378378378, 'f1': 0.7560975609756098}


Processed prompts:   0%|          | 0/168 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

100%|██████████| 50/50 [21:22<00:00, 25.65s/it]

9_table
{'TP': 53, 'FP': 41, 'FN': 21, 'precision': 0.5638297872340425, 'recall': 0.7162162162162162, 'f1': 0.6309523809523809}


,DOC_NAME,STRATEGY,PAIR_F1,PAIR_REC,PAIR_PRC,TP,FP,FN
0,1,doc,0.908046,0.887640,0.929412,79,6,10
1,1,page,0.908046,0.887640,0.929412,79,6,10
2,1,qwen-8b-ft,0.770270,0.640449,0.966102,57,2,32
3,1,row,0.891304,0.921348,0.863158,82,13,7
4,1,table,0.807882,0.921348,0.719298,82,32,7
5,2,doc,0.848684,0.777108,0.934783,129,9,37
6,2,page,0.684932,0.602410,0.793651,100,26,66
7,2,qwen-8b-ft,0.864865,0.771084,0.984615,128,2,38
8,2,row,0.671186,0.596386,0.767442,99,30,67
9,2,table,0.673540,0.590361,0.784000,98,27,68


In [ ]:
!zip --q -r ./gt_old_debug.zip ./gt_old_debug